In [27]:
import pandas as pd 
import torch 
import torch.nn as nn 
import torch.optim as optim 
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from konlpy.tag import Komoran
from collections import Counter
# tqdm : 진행 상태를 로그로 표시하는 기능
from tqdm import tqdm

In [66]:
df = pd.read_csv("../data/ratings_train.txt", sep='\t')

df.dropna(inplace=True)

df.drop_duplicates('document', inplace=True)

df = df[:10000]

len(df)

10000

In [67]:
df['label'].value_counts()

label
0    5027
1    4973
Name: count, dtype: int64

In [68]:
komoran = Komoran()
tokenized_sentence = [komoran.morphs(text) for text in df['document']]

In [69]:
# 단어 사전을 생성 
# 패딩 토큰, 언노운 토큰 생성 (초기 값)
vocab = {
    "<PAD>" : 0, 
    "<UNK>" : 1
}
# tokenized_sentence에서 모든 토큰을 하나의 리스트로 생성 
all_tokens = [ token for tokens in tokenized_sentence for token in tokens ]
# token들의 빈도수를 확인 -> min_count로 제한 
token_counts = Counter(all_tokens)
token_counts

Counter({'.': 7886,
         '이': 6869,
         '하': 6549,
         'ㄴ': 4880,
         '는': 4375,
         '다': 3610,
         '영화': 3560,
         '고': 3121,
         '보': 3082,
         '도': 2232,
         '의': 2214,
         '가': 2204,
         '에': 2161,
         '을': 2065,
         '은': 2062,
         '았': 2022,
         '게': 1883,
         '었': 1723,
         '...': 1708,
         'ㄹ': 1655,
         '어': 1509,
         '들': 1500,
         ',': 1462,
         '지': 1425,
         '아': 1379,
         '를': 1153,
         '?': 1127,
         '없': 1076,
         '있': 1051,
         'ㅁ': 1049,
         '나': 1042,
         '되': 897,
         '만': 810,
         '~': 802,
         '!': 801,
         '는데': 796,
         '주': 740,
         '것': 727,
         '좋': 704,
         '정말': 670,
         '기': 664,
         '적': 653,
         '너무': 642,
         '점': 612,
         '으로': 609,
         '음': 592,
         'ㄴ다': 580,
         '같': 575,
         '안': 574,
         '재밌': 561,
         '

In [70]:
# 단어의 빈도수가 3이상인 토큰들만을 이용하여 단어 사전에 넣어준다. 
for token, count in token_counts.items():
    if count >= 3:
        vocab[token] = len(vocab)

In [71]:
vocab

{'<PAD>': 0,
 '<UNK>': 1,
 '아': 2,
 '더빙': 3,
 '.': 4,
 '진짜': 5,
 '짜증': 6,
 '나': 7,
 '네요': 8,
 '목소리': 9,
 '흠': 10,
 '...': 11,
 '포스터': 12,
 '보고': 13,
 '초딩': 14,
 '영화': 15,
 '줄': 16,
 '....': 17,
 '오버': 18,
 '연기': 19,
 '조차': 20,
 '가볍': 21,
 '지': 22,
 '않': 23,
 '구나': 24,
 '이야기': 25,
 '이': 26,
 '..': 27,
 '솔직히': 28,
 '재미': 29,
 '는': 30,
 '없': 31,
 '다': 32,
 '평점': 33,
 '조정': 34,
 '스럽': 35,
 'ㄴ': 36,
 '가': 37,
 '돋보이': 38,
 '었': 39,
 '던': 40,
 '!': 41,
 '스파이더맨': 42,
 '에서': 43,
 '늙': 44,
 '어': 45,
 '보이': 46,
 '기': 47,
 '만': 48,
 '하': 49,
 '았': 50,
 '너무나': 51,
 '도': 52,
 '막': 53,
 '떼': 54,
 '3': 55,
 '세': 56,
 '부터': 57,
 '초등학교': 58,
 '1': 59,
 '학년': 60,
 '생': 61,
 '아깝': 62,
 'ㅁ': 63,
 '원작': 64,
 '의': 65,
 '긴장감': 66,
 '을': 67,
 '제대로': 68,
 '살리': 69,
 '내': 70,
 '못하': 71,
 '별': 72,
 '반개': 73,
 '욕': 74,
 '나오': 75,
 'ㄴ다': 76,
 '생활': 77,
 '몇': 78,
 '년': 79,
 'ㄴ지': 80,
 '정말': 81,
 '발로': 82,
 '아도': 83,
 '그것': 84,
 '보다': 85,
 '납치': 86,
 '반복': 87,
 '드라마': 88,
 '가족': 89,
 '사람': 90,
 '모이': 91,
 '엇': 92,
 '

In [72]:
# vocab을 이용한 토큰화 된 데이터의 인코딩과 Dataset을 결합 
# dict.get() -> 특정 키를 입력하면 해당 키의 값을 되돌려주는 함수
# ( 두번째 인자값을 이용하여 첫번째 인자의 키 값이 존재하지 않을때 디폴트 값을 설정 )
vocab.get('마케팅', vocab['<UNK>'])

1

In [73]:
# Dataset을 선언
class RNNDataset(Dataset):
    # 생성자, 길이 출력함수, 특정위치의 데이터 출력함수 
    def __init__( self, tokenized_texts, labels, vocab ):
        # tokenized_texts : 토큰화된 문서들 (독립 변수)
        # labels : 정답 데이터 (종속 변수)
        # vocab : 단어 사전 
        self.labels = labels.values
        self.data = [
            [
                vocab.get(token, vocab['<UNK>']) for token in tokens
            ]
            for tokens in tokenized_texts
        ]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        # getitem 함수의 역할 : DataLoader가 데이터를 불러오는 함수 (독립 변수, 종속 변수)
        return torch.tensor(self.data[idx], dtype=torch.long), \
            torch.tensor(self.labels[idx], dtype=torch.long)

In [74]:
# 후 처리 가공 함수 (DataLoader가 배치 사이즈만큼 Dataset을 불러온 후 처리 가공)
def collate_fn(batch):
    # 배치 단위로 들어온 데이터를 최대 길이의 data에 맞게 패딩 토큰을 채워준다. 
    # 배치 -> [ (data, label), (data, label), ... ]
    text_list = [item[0] for item in batch]
    label_list = [item[1] for item in batch]

    # text_list에 있는 인코딩된 데이터에서 최대 길이만큼 나머지 데이터에 패딩 토큰을 채워준다. 
    padded_texts = pad_sequence(text_list, batch_first=True, padding_value=vocab['<PAD>'])
    labels = torch.tensor(label_list, dtype = torch.long)

    return padded_texts, labels

In [75]:
# Dataset 생성 
dataset = RNNDataset(tokenized_sentence, df['label'], vocab)
# train의 길이와 test의 길이를 설정 
train_size = int(len(dataset) * 0.8)   # int() 사용하는 이유는? 길이를 의미하기 때문에 정수형으로 변환(버림)
test_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, test_size])

In [76]:
print(len(train_dataset), len(val_dataset))

8000 2000


In [77]:
train_loader = DataLoader(train_dataset, batch_size = 64, shuffle = True, collate_fn= collate_fn)
val_loader = DataLoader(val_dataset, batch_size = 64, shuffle=True, collate_fn=collate_fn)

In [78]:
class RNNCLF(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size, num_classes):
        # vocab_size : 임베딩 함수 입력 차원의 수 
        # emb_dim : 임베딜 함수 출력 차원의 수 
        # hidden_size : RNN 은닉층의 출력 차원의 수
        # num_classes : 선형 모델의 출력 차원의 수 (분류 개수) 
        super().__init__()
        # 입력되는 데이터는 인코딩 된 데이터 (2,3,4) -> 벡터화 작업 ( nn.Enbedding(), Word2Vec, FastText, Doc2Vec )
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=vocab['<PAD>'])
        # RNN 모델 
        self.rnn = nn.RNN(emb_dim, hidden_size, batch_first=True)
        # 선형 모델 
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        # x : DataLoader의 독립 변수 값(토큰화 데이터)
        embedding = self.emb(x)  # [batch_size, seq_len, emb_dim]

        # rnn_out -> [batch_size, seq_len, hidden_size] (모든 시점의 출력)
        # hidden -> [1, seq_len, hidden_size] (제일 마지막 시점의 은닉 상태)
        rnn_out, hidden = self.rnn(embedding)

        # 선형 모델에 데이터를 대입 하기 위해서 hidden의 배치층을 제거 
        last_hidden = hidden.squeeze(0)  # [seq_len, hidden]

        return self.fc(last_hidden)

In [79]:
# 모델 생성 
model = RNNCLF(len(vocab), emb_dim=64, hidden_size=128, num_classes=2)
# 손실 함수 
criterion = nn.CrossEntropyLoss()
# 옵티마이저 생성 
optimtizer = optim.Adam(model.parameters(), lr = 0.001)


In [80]:
epochs = 50

for epoch in range(epochs):
    model.train()
    train_loss = 0
    corret_train = 0
    total_train = 0
    # tqdm() -> desc는 로그 출력 값
    for inputs, labels in tqdm(train_loader, desc = f"Epoch {epoch+1} / {epochs} Train"):
        optimtizer.zero_grad()
        output = model(inputs)
        loss = criterion(output, labels)
        loss.backward()
        optimtizer.step()

        train_loss += loss.item()
        pred = torch.argmax(output, dim=1)
        corret_train += (pred == labels).sum().item()
        total_train += labels.size(0)
    
    train_acc = (corret_train / total_train) * 100
    avg_train_loss = train_loss / len(train_loader)

    # 검증 구간 
    model.eval()
    val_loss = 0
    corret_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            output = model(inputs)
            loss = criterion(output, labels)

            val_loss += loss.item()
            pred = torch.argmax(output, dim=1)
            corret_val += (pred == labels).sum().item()
            total_val += labels.size(0)
    val_acc = (corret_val / total_val) * 100
    avg_val_loss = val_loss / len(val_loader)
    if (epoch + 1) % 10 == 0:
        print(f"RNN 에폭 : Train Loss : {round(avg_train_loss, 4)} Train Acc : {train_acc}")
        print(f"RNN 에폭 : Vali Loss : {round(avg_val_loss, 4)} Vali Acc : {val_acc}")



Epoch 10 / 50 Train: 100%|██████████| 125/125 [00:01<00:00, 66.90it/s]


RNN 에폭 : Train Loss : 0.6914 Train Acc : 50.2
RNN 에폭 : Vali Loss : 0.696 Vali Acc : 49.05


Epoch 20 / 50 Train: 100%|██████████| 125/125 [00:17<00:00,  7.19it/s]


RNN 에폭 : Train Loss : 0.6964 Train Acc : 50.525
RNN 에폭 : Vali Loss : 0.6936 Vali Acc : 48.75


Epoch 30 / 50 Train: 100%|██████████| 125/125 [00:18<00:00,  6.90it/s]


RNN 에폭 : Train Loss : 0.6967 Train Acc : 50.625
RNN 에폭 : Vali Loss : 0.7046 Vali Acc : 47.3


Epoch 40 / 50 Train: 100%|██████████| 125/125 [00:03<00:00, 41.10it/s]


RNN 에폭 : Train Loss : 0.6966 Train Acc : 50.712500000000006
RNN 에폭 : Vali Loss : 0.6943 Vali Acc : 50.849999999999994


Epoch 50 / 50 Train: 100%|██████████| 125/125 [00:02<00:00, 49.84it/s]


RNN 에폭 : Train Loss : 0.695 Train Acc : 51.1625
RNN 에폭 : Vali Loss : 0.6961 Vali Acc : 49.65
